# NWF Quickstart

Minimal workflow: synthetic 2D data, Charge, Field, k-NN, AgreementRatio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nwf import AgreementRatio, Charge, Field

rng = np.random.RandomState(42)
k = 5

In [ ]:
# Generate 2 classes
X_a = rng.randn(100, 2) * 0.5 + np.array([-1.0, 0.0])
X_b = rng.randn(100, 2) * 0.5 + np.array([1.0, 0.0])
X_train = np.vstack([X_a, X_b])
y_train = np.array([0] * 100 + [1] * 100)

sigma_fixed = np.full(2, 0.1)
field = Field()
for i in range(len(X_train)):
    ch = Charge(z=X_train[i].astype(np.float64), sigma=sigma_fixed.astype(np.float64))
    field.add(ch, labels=[int(y_train[i])], ids=[i])

In [ ]:
# Classify test points
X_test = rng.randn(50, 2) * 0.5
y_test = (X_test[:, 0] > 0).astype(int)

ar = AgreementRatio()
correct = 0
for i in range(len(X_test)):
    q = Charge(z=X_test[i].astype(np.float64), sigma=sigma_fixed.astype(np.float64))
    dist, idx, labs = field.search(q, k=k)
    votes = np.bincount(np.array(labs[0]).astype(int), minlength=2)
    pred = int(np.argmax(votes))
    conf = ar.predict(labs[0], pred)
    if pred == y_test[i]:
        correct += 1

acc = correct / len(X_test)
print(f"Accuracy (k={k}): {acc:.3f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_a[:, 0], X_a[:, 1], c='C0', alpha=0.6, label='Class A')
plt.scatter(X_b[:, 0], X_b[:, 1], c='C1', alpha=0.6, label='Class B')
plt.scatter(X_test[:, 0], X_test[:, 1], c='gray', alpha=0.4, s=20, label='Test')
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('x1'); plt.ylabel('x2'); plt.title(f'NWF Quickstart, accuracy={acc:.2%}')
plt.legend(); plt.show()

Run full script: `python examples/quickstart.py --k 5`